# Page 5 — "How to Reason": the prompt ladder

Five prompt formulations on one model, measuring what each buys — from a forced blurt to
few-shot chain-of-thought. Everything page 5 renders is produced here; the page replays
committed logs and runs no model.

**The one constraint: thinking is OFF for every generation.** The prompt is the only variable —
the *Proof that thinking is off* cell shows the rendered template, and the integrity cell
asserts no trace ever contained thinking content.

Writes to `out/page_05/`: `traces.jsonl` (every generation, published as a download),
`derived.json` (what the page plots, bundled), `manifest.json` (provenance).

All rationale — the rung design, the item set, what the page may and may not claim — lives in
[`docs/page_5_refactor.md`](../docs/page_5_refactor.md).

## How to run this

Two modes, set by one variable in the next cell.

| Mode | Model | Scored (L4-5) | Exemplar pool (L1-2) | Cap | Where it finishes |
|---|---|---|---|---|---|
| `QUICK` | Qwen3-0.6B | 24 | 12 | 512 tokens | a laptop CPU, ~1 h |
| `FULL` | Qwen3-4B | 160 | 24 | 1024 tokens | a 16 GB GPU, ~2 hours |

The two item bands are drawn from opposite ends of MATH-500's difficulty scale: we score on
hard problems and demonstrate with easy ones. Runtimes are estimates until the approval run
measures them, and the loading cell reports a tokens-per-second figure for this machine. `QUICK` is the same pipeline with
noisier bars; the page ships `FULL`. `QUICK` guarantees the aggregates in figure 2, but
figure 1's specimen search can miss on a small model and will fall back with a flag. The run is resumable: `traces.jsonl` is appended as it
goes, and re-running skips work already recorded. Run from this notebook's directory (Jupyter's
default) — outputs land in `out/page_05/` relative to the working directory.

```bash
pip install "torch>=2.7" "transformers>=4.56" accelerate sympy matplotlib requests
pip install reasoning-from-scratch   # or: pip install git+https://github.com/rasbt/reasoning-from-scratch.git
```

In [ ]:
# ---------------------------------------------------------------------------
# The two knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "QUICK"        # "QUICK" (laptop CPU, ~30 min) or "FULL" (GPU, ~2 h)
SEED = 123                # only selects items; decoding is greedy and unseeded

PRESETS = {
    "QUICK": dict(
        model_id="Qwen/Qwen3-0.6B",
        n_eval=24,                     # evaluated items (12 was too thin for figure 1)
        n_pool=12,                     # held-out pool the exemplars are harvested from
        few_shot_k=2,
        max_new_tokens=512,
    ),
    "FULL": dict(
        model_id="Qwen/Qwen3-4B",
        n_eval=160,
        n_pool=24,
        few_shot_k=3,
        max_new_tokens=1024,
    ),
}
CFG = PRESETS[RUN_MODE]

# One generation cap for every rung. This page's variable is the PROMPT, so the
# length budget must not also vary -- that is page 6's experiment. The cap is set
# generously so it should rarely bind; the sweep reports the truncation rate, and
# if it is above a few percent the cap is too low and should be raised.
MAX_NEW_TOKENS = CFG["max_new_tokens"]

# Fallback ladder if the model does not fit: 4B -> 1.7B -> 0.6B. All three are
# hybrid Qwen3 checkpoints with the same tokenizer, so token counts stay
# comparable across them.
#   Qwen3-4B    ~8.0 GB bf16
#   Qwen3-1.7B  ~3.4 GB bf16
#   Qwen3-0.6B  ~1.2 GB bf16

print(f"mode={RUN_MODE}  model={CFG['model_id']}  "
      f"items={CFG['n_eval']} (+{CFG['n_pool']} held out)  cap={MAX_NEW_TOKENS}")

In [ ]:
import hashlib
import json
import random
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# The inherited harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_05")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = lambda xs: sum(xs) / max(len(xs), 1)

print("torch", torch.__version__)

## The grading harness

Inherited unchanged from ch03: `extract_final_candidate` pulls the last `\boxed{...}` out of a
completion (falling back to the last number), and `grade_answer` checks equivalence with light
LaTeX canonicalisation and SymPy. The next cell asserts it behaves as the sweep assumes,
including the "wrote a chain but boxed the answer" case the CoT rungs depend on.

In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                          "9",              True),
    (r"Step 1: 23-20=3. Step 2: 3+6=9. So \boxed{9}.",     "9",              True),   # a chain
    (r"27",                                                 "9",              False),  # a wrong blurt
    (r"\boxed{\frac{1}{2}}",                               "0.5",            True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                     "1024",           True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",           r"\left( 3, \frac{\pi}{2} \right)", True),
    (r"I am not sure.",                                     "9",              False),  # no answer at all
]

rows, failures = [], []
for text, truth, expected in CHECKS:
    got = bool(grade_answer(extract_final_candidate(text), truth))
    rows.append((text, truth, expected, got))
    if got != expected:
        failures.append((text, truth, expected, got))

if failures:                                   # the table is for debugging, not for reading
    for text, truth, expected, got in rows:
        flag = "ok " if got == expected else "FAIL"
        print(f"{flag} {extract_final_candidate(text)!r:<40} vs {truth!r:<12} -> {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")

print(f"grader preflight: {len(rows)}/{len(rows)} as expected "
      f"(boxed, chain-then-boxed, wrong blurt, LaTeX vs decimal, separators, tuples, no answer)")


## The item set: two bands, drawn once

Not a check: this cell *defines the experiment*. It picks the problems, splits them, and hashes
the split so that "the same items" is checkable later rather than asserted.

**What we score: levels 4 and 5.** Full MATH-500 sits near a maths-tuned model's ceiling, and a
ladder whose every rung scores 90% measures nothing.

**What we demonstrate with: levels 1 and 2.** The few-shot rungs prepend worked examples, and a
worked example teaches the *shape* of a solution. It does not have to be as hard as the question,
and in ordinary few-shot chain-of-thought practice it is not. Easy problems also give short
exemplars, which matters because the page renders one verbatim in its "see full prompt"
disclosure, and they are ones a small model can actually solve, which is what makes the harvest
possible at all.

Drawing the two bands from opposite ends of the difficulty scale also makes them **disjoint by
construction**, so a worked answer can never appear in the prompt of a problem we then score.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

EVAL_LEVELS = (4, 5)          # what the ladder is scored on: hard, so there is room to climb
EXEMPLAR_LEVELS = (1, 2)      # where worked examples come from: easy, short, and solvable


def draw(levels, n, seed_offset):
    """A deterministic sample from one difficulty band."""
    band = sorted((r for r in math500 if r["level"] in levels),
                  key=lambda r: r["unique_id"])       # sort before shuffling, or the
    random.Random(SEED + seed_offset).shuffle(band)   # draw depends on dict ordering
    return band[:n]


eval_items = draw(EVAL_LEVELS, CFG["n_eval"], 0)
pool_items = draw(EXEMPLAR_LEVELS, CFG["n_pool"], 1)

# Disjoint by construction (different levels), but assert it anyway: if the bands are
# ever changed to overlap, the few-shot rungs would be scored on problems whose worked
# answers sit in their own prompt.
assert not ({r["unique_id"] for r in pool_items} & {r["unique_id"] for r in eval_items}), \
    "exemplar pool leaked into the eval set"
assert len(eval_items) == CFG["n_eval"], "not enough problems in the eval band"


def sha256_of(strings):
    h = hashlib.sha256()
    for s in strings:
        h.update(s.encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


ITEM_SET_HASH = sha256_of(r["unique_id"] for r in eval_items)

band = lambda items: "+".join(f"L{n}x{sum(r['level'] == n for r in items)}"
                              for n in sorted({r["level"] for r in items}))
print(f"scored    {len(eval_items):>3} items  {band(eval_items):<10} sha256 {ITEM_SET_HASH[:12]}")
print(f"exemplars {len(pool_items):>3} items  {band(pool_items):<10} held out, never scored")


In [ ]:
device = get_device()

# bf16 on GPU keeps quantisation out of our numbers (on Apple MPS it needs the
# torch >= 2.7 the install line pins); fp32 on CPU, where bf16 is slow and patchily
# supported. The dtype actually used is recorded in the manifest.
DTYPE = torch.float32 if device.type == "cpu" else torch.bfloat16

# from_pretrained only draws a progress bar when a checkpoint is split into several
# shards, and these models are a single file -- so the cell would otherwise sit blank
# and you could not tell downloading from loading from hung. Four timed stages instead.

print(f"[1/4] fetch      {CFG['model_id']}")
t0 = time.time()
local_path = snapshot_download(CFG["model_id"])      # draws its own per-file bars
t_fetch = time.time() - t0
print(f"      {t_fetch:>6.1f}s  (cached after the first run)")

print("[2/4] load       tokenizer and weights into RAM")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(local_path)
model = AutoModelForCausalLM.from_pretrained(local_path, dtype=DTYPE, device_map=None)
t_load = time.time() - t0
print(f"      {t_load:>6.1f}s")

print(f"[3/4] move       to {device}")
t0 = time.time()
model = model.to(device).eval()
t_move = time.time() - t0
print(f"      {t_move:>6.1f}s")

# A short generation before the sweep starts. It absorbs the one-off cost of kernel
# compilation and cache allocation, which would otherwise be charged to the first real
# item, and it measures a tokens-per-second figure on THIS machine -- which beats the
# runtime estimates in the README, written without having run anything.
print("[4/4] warm up    8 tokens")
t0 = time.time()
with torch.inference_mode():
    _warm = model.generate(
        **tokenizer("Hello", return_tensors="pt").to(device),
        max_new_tokens=8, do_sample=False,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id,
    )
t_warm = time.time() - t0
WARMUP_TOK_S = 8 / max(t_warm, 1e-6)
print(f"      {t_warm:>6.1f}s  ~{WARMUP_TOK_S:.1f} tokens/s")

n_params = sum(p.numel() for p in model.parameters())
print(f"\nready: {CFG['model_id']}  {n_params/1e9:.2f}B params  {DTYPE}  on {device}")


## Proof that thinking is off

This is the cell to read if you suspect we cheated.

Qwen3's chat template takes an `enable_thinking` flag. Switched **off**, it hands the model a
think block that is *already closed*, so there is nowhere to put a chain and anything the model
writes is its answer. Every generation below uses `enable_thinking=False`, so the prompt is the
only thing that varies across the ladder.

The two renderings are identical until the very end, so only the tail is printed, with newlines
shown as `\n` because the pre-closing is a newline arrangement and would otherwise be invisible.


In [ ]:
demo = [{"role": "user", "content": "What is 2+2?"}]
render = lambda flag: tokenizer.apply_chat_template(
    demo, tokenize=False, add_generation_prompt=True, enable_thinking=flag)

off, on = render(False), render(True)
HAS_THINKING_TOGGLE = off != on

# Everything before the first difference is the same boilerplate in both, so find where
# they diverge and print only from there.
d = 0
while d < min(len(off), len(on)) and off[d] == on[d]:
    d += 1

show = lambda s: s[d:].replace("\n", "\\n")      # newlines are the point here, so make them visible
print(f"both templates agree for {d} characters, ending: ...{off[max(0, d - 30):d]!r}")
print()
print(f"  OFF (used here)  ...{show(off)}")
print(f"  ON  (never used) ...{show(on)}")

if HAS_THINKING_TOGGLE:
    # By structure, not by string inequality: the OFF template must end on a think block
    # that opens and immediately closes.
    assert re.sub(r"\s+", "", off).endswith("<think></think>"), (
        "thinking-off did not pre-close the think block, so this notebook's one constraint "
        "is not actually being enforced by the template")
    assert "<think>" not in on, "thinking-on unexpectedly pre-fills a think block"
    print("\nOK   the OFF template ends on an already-closed think block: the model has")
    print("     nowhere to reason, so whatever it writes next is its answer.")
else:
    print("\nNOTE this checkpoint has no thinking toggle. The constraint still holds -- there")
    print("     is no thinking mode to leave on -- but the manifest records that we did not")
    print("     disable anything, rather than implying we did.")

# End-of-turn ids, derived once here and used by the generation loop below. A model that
# stops on its own emits one of these; it is not visible content and must not be counted
# as a generated token, because token counts are this page's cost axis.
def _id_of(t):
    i = tokenizer.convert_tokens_to_ids(t)
    return i if isinstance(i, int) and i >= 0 and tokenizer.convert_ids_to_tokens(i) == t else None


EOS_IDS = {i for i in (tokenizer.eos_token_id, _id_of("<|im_end|>"), _id_of("<|endoftext|>"))
           if i is not None}
print(f"\nend-of-turn ids: {sorted(EOS_IDS)}")


## The generation loop

The only place a model is called. Everything below this cell is bookkeeping around it.

Three details a re-deriver should not drop. **Token strings**, decoded so that joining them
reproduces the text exactly (asserted), because the page reveals the trace one token at a time
and a character-by-character animation would misrepresent generation on a page about tokens.
**Truncation**, recorded because on this page a binding cap is a defect rather than a variable.
And **end-of-turn tokens are stripped**: they are how the model signals it is finished, not
something it wrote, so counting them would inflate every token count by one.


In [ ]:
def incremental_token_strings(ids):
    """One string per token, such that "".join(pieces) == tokenizer.decode(ids).

    Decoding each token by itself is not safe: the tokenizer is byte-level, so a single
    character (a degree sign, a pi) can be split across two tokens and decoding half of
    one yields a replacement character. Decoding the growing prefix instead and taking
    what each step added avoids that.
    """
    pieces, prev = [], ""
    for k in range(1, len(ids) + 1):
        cur = tokenizer.decode(ids[:k], skip_special_tokens=True)
        pieces.append(cur[len(prev):])
        prev = cur
    return pieces


@torch.inference_mode()
def generate(prompt_text, mdl=None, tok=None, max_new_tokens=None):
    mdl = mdl or model
    tok = tok or tokenizer
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS

    # 1. Wrap the prompt in the model's chat format. enable_thinking=False is the one
    #    constraint of this page: the prompt is the only thing allowed to vary.
    chat = tok.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    # add_special_tokens=False because the template already carries its own special
    # tokens as text; letting the tokenizer add more would prepend a second start marker.
    enc = tok(chat, return_tensors="pt", add_special_tokens=False).to(mdl.device)

    # 2. Greedy decoding. Qwen3 ships sampling defaults in its generation_config, so
    #    do_sample=False is not enough on its own -- the sampling parameters have to be
    #    cleared too, or the manifest would record a setup that was never configured.
    out = mdl.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False, temperature=None, top_p=None, top_k=None,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
    )

    # 3. Keep only what was generated; the prompt is not output.
    new_ids = out[0][enc["input_ids"].shape[1]:].tolist()

    # 4. Whether the cap bound is decided BEFORE stripping, on the raw count. Generate
    #    returns fewer tokens than the cap only if the model stopped on its own.
    truncated = len(new_ids) >= max_new_tokens
    stopped_on_eos = bool(new_ids) and new_ids[-1] in EOS_IDS
    while new_ids and new_ids[-1] in EOS_IDS:
        new_ids.pop()                       # a stop signal, not content the model wrote

    # 5. Decode twice -- once whole, once token by token -- and require agreement. The
    #    page animates these token strings, so a silent mismatch would put a lie on screen.
    text = tok.decode(new_ids, skip_special_tokens=True)
    tokens = incremental_token_strings(new_ids)
    assert "".join(tokens) == text, "token strings do not reconstruct the decoded text"

    return {
        "text": text,
        "tokens": tokens,
        "tokens_generated": len(new_ids),
        "truncated": truncated,
        "stopped_on_eos": stopped_on_eos,
        "thinking_leaked": "<think>" in text,
    }


# A smoke test has one job: tell a broken harness apart from a weak model. So it shows
# the prompt as actually sent, the ids beside what they decode to, and why generation
# stopped. Two probes -- if both are wrong the harness is suspect; if only the harder
# one is wrong, that is a 0.6B being a 0.6B, which is information rather than a fault.
for question, expect in [("What is 2 + 2?", "4"), ("What is 10 - 3?", "7")]:
    g = generate(question, max_new_tokens=32)
    stop = "end-of-turn" if g["stopped_on_eos"] else ("hit the cap" if g["truncated"] else "?")
    verdict = "as expected" if expect in g["text"] else f"NOT {expect!r}"
    print(f"{question!r:<20} -> {g['text']!r:<24} {g['tokens_generated']:>2} tokens, "
          f"stopped: {stop:<12} {verdict}")

print("\n--- the last probe in full " + "-" * 46)
print("prompt as sent:")
print(tokenizer.apply_chat_template([{"role": "user", "content": "What is 10 - 3?"}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False).replace("\n", "\\n"))
ids = tokenizer(g["text"], add_special_tokens=False).input_ids
print("\ntokens:", [(i, s) for i, s in zip(ids, g["tokens"])][:12])


## The ladder

Five formulations, from least to most instruction. Rung design rationale is in the spec, §§2–3.

| Rung | What is added | Whose mechanism |
|---|---|---|
| `bare` | nothing — the harness prompt alone | the true default (ch03's baseline) |
| `forced_blurt` | "Give only the final answer. Do not explain." | the floor |
| `zero_shot_cot` | `"\n\nExplain step by step."` | Kojima et al. 2022 · ch04 §4.3 |
| `one_shot` | one worked example prepended | Wei et al. 2022 |
| `few_shot` | k worked examples prepended | Wei et al. 2022 |

**Every rung, including `bare`, carries ch03's `render_prompt`, which fixes the `\boxed{}`
output format.** Vary that too and the ladder is confounded — a rung could score lower only
because the grader cannot find its answer. The exemplar rungs get no "explain step by step":
combining Wei's and Kojima's mechanisms would measure a third thing.

In [ ]:
RUNGS = [
    ("bare",          "bare question"),
    ("forced_blurt",  "\"answer with the number\""),
    ("zero_shot_cot", "\"explain step by step\""),
    ("one_shot",      "one worked example"),
    ("few_shot",      f"{CFG['few_shot_k']} worked examples"),
]
RUNG_KEYS = [k for k, _ in RUNGS]
RUNG_LABEL = dict(RUNGS)

FORCED_BLURT_SUFFIX = "\n\nGive only the final answer. Do not explain."
COT_SUFFIX = "\n\nExplain step by step."      # ch04 section 4.3, verbatim mechanism


def build_prompt(rung, question, exemplars=None):
    '''The whole ladder, in one readable function. Nothing else varies.'''
    base = render_prompt(question)             # ch03's template -- CONSTANT across rungs

    if rung == "bare":
        return base
    if rung == "forced_blurt":
        return base + FORCED_BLURT_SUFFIX
    if rung == "zero_shot_cot":
        return base + COT_SUFFIX

    k = 1 if rung == "one_shot" else CFG["few_shot_k"]
    shots = "".join(
        render_prompt(e["question"]) + "\n" + e["trace"].strip() + "\n\n"
        for e in (exemplars or [])[:k]
    )
    return shots + base


# The prompts ARE the experiment on this page: the five rungs differ in nothing else.
# So print the shared stem once, then only what each rung adds and where. Exemplars go
# BEFORE the stem, instructions go after it -- the same split figure 1 uses to highlight
# the diff, so the notebook and the page describe a prompt the same way.
STEM = render_prompt("<QUESTION>")

print("the stem, identical in all five rungs")
print("-" * 74)
print(STEM)
print("-" * 74)

for key, label in RUNGS:
    p = build_prompt(key, "<QUESTION>", exemplars=None)
    before, after = p[:p.index(STEM)], p[p.index(STEM) + len(STEM):]
    if key in ("one_shot", "few_shot"):
        k = 1 if key == "one_shot" else CFG["few_shot_k"]
        note = f"prepends {k} worked example{'s' if k > 1 else ''} (harvested later, printed in full there)"
    elif after:
        note = f"appends {after.strip()!r}"
    else:
        note = "adds nothing"
    print(f"  {key:<14} {note}")

print(f"\nthe {len(COT_SUFFIX.split())} words figure 1 highlights: {COT_SUFFIX.strip()!r}")


## The sweep, part 1: the machinery

This cell defines one function. Hand it a rung and a list of problems, and it asks the model each
question in turn, records what came back, and gives you the records.

Two practical details. It writes every answer to `traces.jsonl` the moment that answer arrives,
so if the run stops halfway you lose nothing: starting again skips whatever is already on disk.
And it keys that file by `(rung, problem)`, so re-running a rung you have already finished costs
nothing at all.

Nothing here talks to the model differently from the smoke test above. It is the same `generate`
call in a loop, with bookkeeping around it.

The record it builds per problem is the schema shared with pages 6 through 9, with `rung` in the
place page 6 uses for `budget`. Provenance is not repeated on every line; it lives once in
`manifest.json`.


In [ ]:
RUN_ID = f"{RUN_MODE.lower()}-{CFG['model_id'].split('/')[-1]}-{ITEM_SET_HASH[:8]}"


def load_done():
    '''(rung, problem_id) -> record, for resume.'''
    done = {}
    if TRACES_PATH.exists():
        with TRACES_PATH.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r.get("run_id") == RUN_ID:
                    done[(r["rung"], r["problem_id"])] = r
    return done


DONE = load_done()
print(f"run_id={RUN_ID}   {len(DONE)} generations already on disk")


def append_trace(rec):
    with TRACES_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def run_rung(rung, items, exemplars=None, mdl=None, tok=None, label=None, split="eval"):
    records, n_new = [], 0
    exemplar_ids = [e["problem_id"] for e in (exemplars or [])][
        : (1 if rung == "one_shot" else CFG["few_shot_k"])
    ] if rung in ("one_shot", "few_shot") else []

    bar = tqdm(items, desc=f"{label or rung:<20}", unit="item")
    for row in bar:
        key = (rung, row["unique_id"])
        if key in DONE:
            records.append(DONE[key])          # already on disk from an earlier run
            bar.set_postfix(acc=f"{sum(r['correct'] for r in records)/len(records):.2f}",
                            new=n_new)
            continue

        prompt = build_prompt(rung, row["problem"], exemplars)
        g = generate(prompt, mdl=mdl, tok=tok)
        extracted = extract_final_candidate(g["text"])

        rec = {
            "run_id": RUN_ID,
            "problem_id": row["unique_id"],
            "split": split,                    # "eval" is scored; "held_out" only feeds exemplars
            "level": row["level"],
            "subject": row["subject"],
            "question": row["problem"],
            "ground_truth": row["answer"],
            "rung": rung,
            "prompt": prompt,
            "trace": g["text"],
            "tokens": g["tokens"],
            "truncated": g["truncated"],
            "extracted_answer": extracted,
            "correct": bool(grade_answer(extracted, row["answer"])),
            "tokens_generated": g["tokens_generated"],
            "exemplar_ids": exemplar_ids,
            "thinking_leaked": g["thinking_leaked"],
        }
        append_trace(rec)                      # written now, so a crash costs nothing
        DONE[key] = rec
        records.append(rec)
        n_new += 1
        bar.set_postfix(acc=f"{sum(r['correct'] for r in records)/len(records):.2f}", new=n_new)

    return records

## The sweep, part 2: run it

Now we use it. Three of the five rungs need no worked examples, so they go first.

The fourth pass is the one doing double duty: the same zero-shot prompt, run over the held-out
easy problems instead of the scored ones. Those answers are not graded against anything. They
are where the worked examples come from in the next section, which is why they have to exist
before the two exemplar rungs can run at all.

At the end, one number decides how the page opens: **does the model already show its working
when nobody asked it to?** If it does, the page's opening rung is the one that forbids
explaining, and its verb becomes *let it reason* rather than *make it reason*. That verdict is
computed here and written into `derived.json`, so it cannot be quietly reversed in prose later.


In [ ]:
# Rough projection from the warm-up, so you know what you signed up for before it starts.
# Generous: it assumes every generation runs to the cap, and most stop well before it.
_gens = len(RUNG_KEYS) * len(eval_items) + len(pool_items)
print(f"about {_gens} generations; at ~{WARMUP_TOK_S:.0f} tok/s and a {MAX_NEW_TOKENS}-token cap "
      f"that is under {_gens * MAX_NEW_TOKENS / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h worst case.\n")

# Phase A: the three rungs that need no exemplars. zero_shot_cot also runs on the
# held-out pool -- that pass is what the worked examples are harvested from.
results = {}
for rung in ("bare", "forced_blurt", "zero_shot_cot"):
    results[rung] = run_rung(rung, eval_items, label=rung)

pool_cot = run_rung("zero_shot_cot", pool_items, label="pool/zero_shot_cot", split="held_out")


# The framing verdict (spec §3): does the bare rung already show working? Decides
# figure 1's left-hand rung and the page's verb ("make it reason" vs "let it
# reason"); recorded in derived.json so it cannot later be reversed in prose.
def shows_working(rec):
    lines = [ln for ln in rec["trace"].strip().splitlines() if ln.strip()]
    return len(lines) > 2 or rec["tokens_generated"] > 60


bare_working_share = mean([shows_working(r) for r in results["bare"]])
BARE_ALREADY_REASONS = bare_working_share >= 0.5
FRAMING_VERB = "let it reason" if BARE_ALREADY_REASONS else "make it reason"
FIGURE1_LEFT = "forced_blurt" if BARE_ALREADY_REASONS else "bare"

print(f"\nbare rung shows working on {bare_working_share:.0%} of items")
print(f'-> page verb: "{FRAMING_VERB}"   figure 1 left: {FIGURE1_LEFT}')

## Harvesting the worked examples

**The exemplars are the model's own verified-correct solutions to held-out problems** —
authored exemplars would be the one thing on the page a reader could not re-derive (spec §3).
Selection is a deterministic filter: correct under the grader, untruncated, short enough to
fit, shortest first with ties broken by `problem_id`.

In [ ]:
MAX_EXEMPLAR_TOKENS = 400

# Report the pool's yield whether or not it worked, so a near miss is visible on the
# run before it becomes a failure on the next one.
n_correct = sum(r["correct"] for r in pool_cot)
print(f"held-out pool: {n_correct}/{len(pool_cot)} solved correctly "
      f"({n_correct / max(len(pool_cot), 1):.0%})")

candidates = [
    r for r in pool_cot
    if r["correct"] and not r["truncated"] and r["tokens_generated"] <= MAX_EXEMPLAR_TOKENS
]
candidates.sort(key=lambda r: (r["tokens_generated"], r["problem_id"]))

if not candidates:
    raise RuntimeError(
        f"No usable exemplars. Of {len(pool_cot)} held-out problems, {n_correct} were solved "
        f"correctly and none of those were both untruncated and <= {MAX_EXEMPLAR_TOKENS} "
        f"tokens.\n\n"
        f"The two levers, in order of preference:\n"
        f"  1. raise CFG['n_pool'] (currently {CFG['n_pool']}) -- exemplars come from levels "
        f"{EXEMPLAR_LEVELS}, so these are cheap generations;\n"
        f"  2. use a larger model.\n\n"
        f"Do not author exemplars by hand. The page's claim is that they are the model's own "
        f"verified work, and that claim is the reason this raises instead of falling back."
    )

exemplars = [
    {
        "problem_id": r["problem_id"],
        "level": r["level"],
        "question": r["question"],
        "trace": r["trace"],
        "ground_truth": r["ground_truth"],
        "extracted_answer": r["extracted_answer"],
        "tokens_generated": r["tokens_generated"],
    }
    for r in candidates[: CFG["few_shot_k"]]
]

if len(exemplars) < CFG["few_shot_k"]:
    print(f"WARNING: only {len(exemplars)} usable exemplars from a pool of {len(pool_items)}.\n"
          f"         Raise n_pool, or MAX_EXEMPLAR_TOKENS, and re-run. The few-shot rung\n"
          f"         below will use however many were found, which weakens the comparison.")

print(f"{len(candidates)} of {len(pool_items)} held-out problems were solved correctly, "
      f"untruncated, and short enough.\n")
for e in exemplars:
    print(f"  {e['problem_id']:<34} L{e['level']}  {e['tokens_generated']:>4} tokens  "
          f"-> {e['extracted_answer']}")

print("\n--- the first exemplar, exactly as it will be prepended " + "-" * 20)
print(render_prompt(exemplars[0]["question"]))
print(exemplars[0]["trace"].strip()[:800])

In [ ]:
# Phase B: the exemplar rungs. Their prompts contain no instruction to reason --
# only examples of reasoning.
for rung in ("one_shot", "few_shot"):
    results[rung] = run_rung(rung, eval_items, exemplars=exemplars, label=rung)

## Sweep integrity

Three assertions the page's honesty rests on. If any of these fails, the numbers below do not
mean what the page says they mean.

In [ ]:
all_records = [r for rung in RUNG_KEYS for r in results[rung]]

leaked = [r for r in all_records if r["thinking_leaked"]]
print(f"1. thinking never leaked into a trace ......... {len(leaked)} violations")
assert not leaked, "a trace contained a <think> block; the page's core constraint is broken"

trunc_rate = sum(r["truncated"] for r in all_records) / len(all_records)
print(f"2. the generation cap rarely bound ............ {trunc_rate:.1%} truncated")
if trunc_rate > 0.05:
    print("   ^ above 5%: raise MAX_NEW_TOKENS and re-run. Page 5 lets every prompt run to")
    print("     natural length; capping it is page 6's experiment, not this one.")

ex_ids = {e["problem_id"] for e in exemplars}
overlap = ex_ids & {r["problem_id"] for r in results["few_shot"]}
print(f"3. exemplars stayed out of the eval set ....... {len(overlap)} overlaps")
assert not overlap, "an exemplar problem was also scored"

print(f"\n{len(all_records)} generations across {len(RUNG_KEYS)} rungs and "
      f"{len(eval_items)} items.")

## Reasoning steps, and where they sit in the token stream

Figure 1 walks a chain one step at a time, so the page needs the trace divided into steps.
That division is a derivation, so it belongs here rather than in TypeScript: doing it during
translation would put logic on the page that the cross-language test cannot check.

The rule, in order of preference: blank-line paragraphs, then newlines, then sentences, taking
the first that actually divides the trace. Fragments shorter than `MIN_STEP_CHARS` merge into
the step before them, so the reader never sees an orphan line. Each step carries its token
range, and the ranges are contiguous and cover every token, so the page's reveal animates real
token boundaries and stops at step ends.


In [ ]:
# A "fragment" is an orphan like "OK" or "So:", not a short-but-real step. Length
# alone cannot tell them apart: "5+2=7" is five characters and carries a whole
# claim. A digit is the signal, which is the same reason page 9 argues a step has
# to be big enough to carry a fact. Two kinds of span are never merged away.
MIN_STEP_CHARS = 12


def _raw_spans(text):
    """First splitting rule that actually divides the trace: paragraphs, lines, sentences."""
    for pattern in (r"\n\s*\n", r"\n", r"(?<=[.!?])\s+"):
        spans, pos = [], 0
        for m in re.finditer(pattern, text):
            if m.start() > pos:
                spans.append((pos, m.start()))
            pos = m.end()
        if pos < len(text):
            spans.append((pos, len(text)))
        spans = [(a, b) for a, b in spans if text[a:b].strip()]
        if len(spans) > 1:
            return spans
    return [(0, len(text))] if text.strip() else []


def segment_steps(text, tokens):
    """Steps with token ranges. Relies on "".join(tokens) == text, asserted at generation."""
    assert "".join(tokens) == text, "token strings do not reconstruct the trace"

    raw = _raw_spans(text)
    spans = []
    for k, (a, b) in enumerate(raw):
        piece = text[a:b].strip()
        is_last = k == len(raw) - 1
        mergeable = (
            len(piece) < MIN_STEP_CHARS
            and not re.search(r"\d", piece)         # no number, so it states nothing
            and not is_last                        # the landing line stays its own step
            and "\\boxed" not in piece             # so does whatever carries the answer
        )
        if spans and mergeable:
            spans[-1] = (spans[-1][0], b)          # merge fragments backwards
        else:
            spans.append((a, b))

    ends, run = [], 0                              # cumulative char length per token
    for t in tokens:
        run += len(t)
        ends.append(run)

    def token_at(char_index):
        lo, hi = 0, len(ends)
        while lo < hi:
            mid = (lo + hi) // 2
            if ends[mid] <= char_index:
                lo = mid + 1
            else:
                hi = mid
        return lo

    steps = [{"text": text[a:b].strip(), "token_start": token_at(a)} for a, b in spans]
    for j, st in enumerate(steps):                 # contiguous ranges covering every token
        st["token_end"] = steps[j + 1]["token_start"] if j + 1 < len(steps) else len(tokens)
    return steps


# Checked against a real trace from the sweep, not a toy string.
_probe = next((r for r in results["zero_shot_cot"] if r["correct"]), results["zero_shot_cot"][0])
_steps = segment_steps(_probe["trace"], _probe["tokens"])
_norm = lambda s: re.sub(r"\s+", "", s)
assert _norm("".join(s["text"] for s in _steps)) == _norm(_probe["trace"]), \
    "segmentation dropped text"
assert _steps[-1]["token_end"] == len(_probe["tokens"]), "segmentation did not cover every token"

print(f"{_probe['problem_id']}: {_probe['tokens_generated']} tokens -> {len(_steps)} steps")
for s in _steps[:6]:
    print(f"  [{s['token_start']:>3}:{s['token_end']:>3}]  {s['text'][:88]}")


## Figure 1 — three panels on one problem

The page opens on the four-word diff and **ends on the worked example**, so it moves from
*tell it* to *show it*, and figure 2 then explains why the second buys more. All three panels
answer the same question with the same weights and the same decoding.

| Panel | Rung | What it shows |
|---|---|---|
| left | the framing verdict's rung (`bare` or `forced_blurt`) | the default: no chain |
| middle | `zero_shot_cot` | four words, and a chain appears |
| right | `one_shot` | a worked example, and the chain takes the exemplar's shape |

**Curated from the sweep, not generated for the figure.** Left must miss and middle must land.
The right panel is a *preference* in the sort, never a filter, so adding it does not narrow the
search. Its caption is about the **shape** of the chain rather than about who got it right:
format imitation is what a worked example demonstrably changes here, and claiming more than
that would outrun the run (spec §6).


In [ ]:
MAX_SPECIMEN_TOKENS = 260      # a chain the reader can walk on screen

left_by_id = {r["problem_id"]: r for r in results[FIGURE1_LEFT]}
one_by_id = {r["problem_id"]: r for r in results["one_shot"]}

triples = []
for mid in results["zero_shot_cot"]:
    left = left_by_id.get(mid["problem_id"])
    if left and not left["correct"] and mid["correct"] and not mid["truncated"]:
        triples.append((left, mid, one_by_id.get(mid["problem_id"])))

# Preferences in order: a usable third panel, a walkable chain, then shortest, then
# problem_id for determinism. None of these is a filter, so the third panel costs
# the search nothing.
def _rank(t):
    _, mid, right = t
    return (
        not (right and right["correct"] and not right["truncated"]),
        mid["tokens_generated"] > MAX_SPECIMEN_TOKENS,
        mid["tokens_generated"],
        mid["problem_id"],
    )


triples.sort(key=_rank)
SPECIMEN_IS_FALLBACK = False

if triples:
    left_spec, mid_spec, right_spec = triples[0]
else:
    # QUICK on a small model can miss by chance. Degrade to the best available
    # illustration and flag it rather than failing the run: figure 2's aggregates
    # are unaffected, and a flagged specimen must not be translated to the page.
    SPECIMEN_IS_FALLBACK = True
    mid_spec = min((r for r in results["zero_shot_cot"] if not r["truncated"]),
                   key=lambda r: (not r["correct"], r["tokens_generated"], r["problem_id"]))
    left_spec = left_by_id[mid_spec["problem_id"]]
    right_spec = one_by_id.get(mid_spec["problem_id"])
    print("WARNING: no problem where the left rung misses and zero-shot CoT lands.\n"
          "         Falling back to an illustrative triple; derived.json flags it.\n"
          "         Raise n_eval or run FULL before translating this to the page.\n")

# Every rung shares ch03's render_prompt stem; suffix rungs add after it, exemplar
# rungs add before it. Splitting on the stem gives the page one highlight rule for
# all three panels instead of a per-rung special case.
PROMPT_STEM = render_prompt(mid_spec["question"])

PANELS = []
for name, rec in (("left", left_spec), ("middle", mid_spec), ("right", right_spec)):
    if rec is None:
        continue
    p = rec["prompt"]
    assert PROMPT_STEM in p, f"{name} panel's prompt does not contain the shared stem"
    cut = p.index(PROMPT_STEM)
    PANELS.append({
        "panel": name,
        "rung": rec["rung"],
        "prompt_before": p[:cut],                       # exemplars, on the one-shot panel
        "prompt_after": p[cut + len(PROMPT_STEM):],     # suffix, on the instruction rungs
        "trace": rec["trace"],
        "tokens": rec["tokens"],
        "steps": segment_steps(rec["trace"], rec["tokens"]),
        "tokens_generated": rec["tokens_generated"],
        "extracted_answer": rec["extracted_answer"],
        "correct": rec["correct"],
    })

print(f"{len(triples)} candidate triples"
      + ("   [FALLBACK SPECIMEN]" if SPECIMEN_IS_FALLBACK else "")
      + f";  chosen {mid_spec['problem_id']} (L{mid_spec['level']})\n")
print("QUESTION:", mid_spec["question"][:300])
print("TRUTH   :", mid_spec["ground_truth"])

for pan in PANELS:
    mark = "correct" if pan["correct"] else "wrong  "
    added = (pan["prompt_after"] or pan["prompt_before"]).strip()
    print(f"\n--- {pan['panel']:<6} {pan['rung']:<14} {mark}  "
          f"{pan['tokens_generated']:>4} tokens, {len(pan['steps'])} steps")
    print(f"    prompt adds: {added[:100]!r}" if added else "    prompt adds: nothing")
    for s in pan["steps"][:4]:
        print("     ", s["text"][:92])
    if len(pan["steps"]) > 4:
        print(f"      ... {len(pan['steps']) - 4} more steps")


## Figure 2 — the ladder, measured

Horizontal bars, one per rung; the cost row underneath is the handoff to page 6. **Model size
appears on no axis** (spec §1).

In [ ]:
def summarise(rung):
    rs = results[rung]
    toks = sorted(r["tokens_generated"] for r in rs)
    return {
        "rung": rung,
        "label": RUNG_LABEL[rung],
        "n": len(rs),
        "n_correct": sum(r["correct"] for r in rs),
        "accuracy": mean([r["correct"] for r in rs]),
        "mean_tokens": mean(toks),
        "median_tokens": toks[len(toks) // 2],
        "total_tokens": sum(toks),
        "truncated": sum(r["truncated"] for r in rs),
    }


summary = [summarise(k) for k in RUNG_KEYS]
acc = {s["rung"]: s["accuracy"] for s in summary}

print(f"{'rung':<16}{'prompt':<26}{'solved':>10}{'accuracy':>10}{'mean tok':>10}")
print("-" * 72)
for s in summary:
    print(f"{s['rung']:<16}{s['label']:<26}{s['n_correct']:>4}/{s['n']:<5}"
          f"{s['accuracy']:>9.1%}{s['mean_tokens']:>10.0f}")

best = max(summary, key=lambda s: s["accuracy"])
floor = min(summary, key=lambda s: s["accuracy"])

# The page's caption is drafted against these, not against expectations.
ordered = [s["rung"] for s in sorted(summary, key=lambda s: s["accuracy"])]
expected = ["forced_blurt", "bare", "zero_shot_cot", "one_shot", "few_shot"]
collapse = abs(acc["few_shot"] - acc["one_shot"]) * 100 < 2.0

print(f"\nranking, worst to best: {' < '.join(ordered)}"
      + ("" if ordered == expected
         else "  (differs from the naive expectation -- the caption should say so)"))
if collapse:
    print("one_shot and few_shot within 2 points -> consider collapsing them into one rung")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.6))
ys = range(len(summary))[::-1]
ax.barh(list(ys), [s["accuracy"] * 100 for s in summary], height=0.55, color="#3b4a6b")
ax.set_yticks(list(ys))
ax.set_yticklabels([s["label"] for s in summary])
ax.set_xlabel("share of problems solved (%)")
ax.set_xlim(0, 100)
ax.spines[["top", "right"]].set_visible(False)

for y, s in zip(ys, summary):
    ax.text(s["accuracy"] * 100 + 1.2, y, f"{s['accuracy']*100:.0f}%",
            va="center", fontsize=9)

cost = "  ·  ".join(f"{s['mean_tokens']:.0f}" for s in summary)
ax.set_title(f"{CFG['model_id']} · thinking off · greedy · MATH-500 L4-5, n={len(eval_items)}",
             fontsize=10, loc="left")
fig.text(0.01, -0.04, f"tokens written, same order:  {cost}", fontsize=9, color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2.png", dpi=160, bbox_inches="tight")
plt.show()

## The close

One paired readout, not a third chart: the best rung's accuracy beside what it cost. This is
the number page 6 opens on.

In [ ]:
close = {
    "best_rung": best["rung"],
    "best_label": best["label"],
    "best_accuracy": best["accuracy"],
    "best_mean_tokens": best["mean_tokens"],
    "floor_rung": floor["rung"],
    "floor_label": floor["label"],
    "floor_accuracy": floor["accuracy"],
    "floor_mean_tokens": floor["mean_tokens"],
    "accuracy_gain_pts": (best["accuracy"] - floor["accuracy"]) * 100,
    "token_multiple": best["mean_tokens"] / max(floor["mean_tokens"], 1e-9),
}
print(f"{close['best_label']:<26} {close['best_accuracy']:>6.1%}   "
      f"{close['best_mean_tokens']:>6.0f} tokens/problem")
print(f"{close['floor_label']:<26} {close['floor_accuracy']:>6.1%}   "
      f"{close['floor_mean_tokens']:>6.0f} tokens/problem")
print(f"\n{close['accuracy_gain_pts']:+.1f} points of accuracy, "
      f"for {close['token_multiple']:.0f}x the tokens.")

## Artifacts

`derived.json` is the contract between this notebook and the page: the page reads it and
nothing else, and **no inference runs at demonstration time.** It carries the aggregates, the
figure 1 specimens with their token strings, the harvested exemplars plus the exact few-shot
prompt for the specimen problem (the "see full prompt" disclosure renders it verbatim), and
the framing verdict. `traces.jsonl` stays out of the web bundle and is published as a download
beside the notebook link.

In [ ]:
def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None


def repo_revision(model_id):
    try:
        from huggingface_hub import model_info
        return model_info(model_id).sha
    except Exception:
        return None


def tokenizer_fingerprint(tok):
    try:
        return hashlib.sha256(tok.backend_tokenizer.to_str().encode()).hexdigest()
    except Exception:
        return None


manifest = {
    "run_id": RUN_ID,
    "notebook": "page_05_how_to_reason.ipynb",
    "page": 5,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": git_sha(),
    "model": {
        "id": CFG["model_id"],
        "revision": repo_revision(CFG["model_id"]),
        "tokenizer_sha256": tokenizer_fingerprint(tokenizer),
        "params": n_params,
        "dtype": str(DTYPE),
        "thinking": "off",
        "has_thinking_toggle": HAS_THINKING_TOGGLE,
    },
    "decoding": {"strategy": "greedy", "do_sample": False,
                 "max_new_tokens": MAX_NEW_TOKENS},
    "items": {
        "source": "MATH-500 (ch03 copy)",
        "filter": "level in {4, 5}",
        "n_eval": len(eval_items),
        "n_held_out": len(pool_items),
        "eval_set_sha256": ITEM_SET_HASH,
        "seed": SEED,
    },
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "chat_template_thinking_off": off,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device)},
    "integrity": {"thinking_leaks": len(leaked), "truncation_rate": trunc_rate},
}

# The exact few-shot prompt sent for the specimen problem, straight from the sweep
# records -- the page's "see full prompt" disclosure renders this verbatim instead
# of re-assembling it in TypeScript.
few_shot_prompt = next(
    r["prompt"] for r in results["few_shot"] if r["problem_id"] == mid_spec["problem_id"]
)

derived = {
    "manifest": manifest,
    "framing": {
        "bare_rung_already_reasons": BARE_ALREADY_REASONS,
        "bare_working_share": bare_working_share,
        "verb": FRAMING_VERB,
        "figure1_left_rung": FIGURE1_LEFT,
    },
    "figure_1": {
        "problem_id": mid_spec["problem_id"],
        "level": mid_spec["level"],
        "question": mid_spec["question"],
        "ground_truth": mid_spec["ground_truth"],
        "prompt_stem": PROMPT_STEM,
        "panels": PANELS,
        "n_candidate_triples": len(triples),
        "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
    },
    "figure_2": {
        "rungs": summary,
        "ranking_worst_to_best": ordered,
        "matches_naive_expectation": ordered == expected,
        "one_shot_few_shot_within_2pts": collapse,
    },
    "close": close,
    "exemplars": exemplars,
    "few_shot_prompt_example": {
        "problem_id": mid_spec["problem_id"],
        "prompt": few_shot_prompt,
    },
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

kb = lambda p: p.stat().st_size / 1024
print(f"derived.json   {kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
print(f"manifest.json  {kb(MANIFEST_PATH):>9.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {kb(TRACES_PATH):>9.1f} KB   <- published as a download")

## Notes for re-derivation

- Greedy decoding can diverge on a handful of items across hardware through floating-point
  differences (`get_device` enables TF32 matmuls on CUDA, one such source): expect the shape
  and the ranking to reproduce, not every digit.
- *n* is small (160 in `FULL`); adjacent rungs within a couple of points are not resolvable.

What the page may and may not claim from this run — and the full limitations list — is
[`docs/page_5_refactor.md`](../docs/page_5_refactor.md) §6, which the translation step checks
itself against ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).